In [ ]:
# Shared Setup: run this cell first before any section
# Committed by: Zoey Le

import pandas as pd

# -----------------------------------------------
# Dataset: Q1 2024 Sales Records
# 20 orders — date, region, product, quantity, price, sales rep
# -----------------------------------------------

sales_raw = {
    'OrderID':   [1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,
                  1011,1012,1013,1014,1015,1016,1017,1018,1019,1020],
    'Date':      ['2024-01-05','2024-01-12','2024-01-18','2024-01-22','2024-01-30',
                  '2024-02-03','2024-02-08','2024-02-14','2024-02-19','2024-02-25',
                  '2024-03-02','2024-03-07','2024-03-11','2024-03-15','2024-03-18',
                  '2024-03-20','2024-03-22','2024-03-25','2024-03-27','2024-03-30'],
    'Region':    ['North','South','East','West','North','East','South','North','West','East',
                  'South','North','East','West','South','North','East','West','South','North'],
    'Product':   ['Laptop','Headset','Monitor','Keyboard','Laptop','Webcam','Headset','Monitor',
                  'Laptop','Keyboard','Webcam','Laptop','Monitor','Headset','Keyboard','Webcam',
                  'Laptop','Monitor','Headset','Keyboard'],
    'Quantity':  [2,5,1,8,1,10,3,2,1,12,7,2,1,4,6,9,1,2,5,10],
    'UnitPrice': [1200.00,89.99,499.00,49.99,1200.00,79.99,89.99,499.00,1200.00,49.99,
                  79.99,1200.00,499.00,89.99,49.99,79.99,1200.00,499.00,89.99,49.99],
    'SalesRep':  ['Jordan','Taylor','Morgan','Jordan','Casey','Taylor','Morgan','Casey',
                  'Jordan','Taylor','Casey','Morgan','Jordan','Taylor','Casey','Morgan',
                  'Jordan','Casey','Taylor','Morgan']
}

df = pd.DataFrame(sales_raw)
df['Date'] = pd.to_datetime(df['Date'])
df['Revenue'] = df['Quantity'] * df['UnitPrice']

print('Dataset loaded successfully.')
print(f'  Orders: {len(df)}')
print(f'  Date range: {df.Date.min().date()} to {df.Date.max().date()}')
print(f'  Total revenue: ${df.Revenue.sum():,.2f}')

### Section 1 Narrative - Zoey Le

The dataset contains business sales data including information about products, regions, 
sales representatives, order quantities, and revenue per order. The dataset summary 
revealed multiple unique products, regions, and sales reps, giving a broad view of 
business performance across different dimensions. The data quality check examined three 
key areas: missing values, duplicate orders, and negative quantities — any warnings 
found should be addressed before drawing conclusions from the analysis. Overall, the 
dataset appears structured and ready for exploratory analysis, though readers should 
keep in mind any data quality warnings flagged above when interpreting the results.

In [ ]:
# Section 1, Cell 1: Explore the dataset
# Run the Shared Setup cell above first

# TODO: Display basic info about the dataset
# Use: df.shape, df.dtypes, df.head(), df.describe()



print('Dataset shape:', df.shape)
print()
print('Column types:')
print(df.dtypes)
print()
print('First 5 rows:')
print(df.head())

# Clear outputs -> Save -> Commit: 'Section 1: Add initial exploration - [Name]'


In [ ]:
# Section 1, Cell 2: Data quality check

def check_data_quality(dataframe):
    """Check the dataset for missing values, duplicates, and out-of-range values.

    Args:
        dataframe: the sales DataFrame

    Returns:
        dict with keys: 'missing_values', 'duplicate_orders', 'negative_quantities'
        Each value is an integer count.

    >>> import pandas as pd
    >>> test_df = pd.DataFrame({'OrderID': [1,2,2], 'Quantity': [5, -1, 3],
    ...                         'Revenue': [100, 50, 75]})
    >>> result = check_data_quality(test_df)
    >>> result['duplicate_orders']
    2
    >>> result['negative_quantities']
    1
    """
    missing_values = dataframe.isnull().sum().sum()
    duplicate_orders = dataframe.duplicated().sum()
    negative_quantities = dataframe[dataframe['Quantity'] < 0].shape[0]

    return {
        'missing_values': missing_values,
        'duplicate_orders': duplicate_orders,
        'negative_quantities': negative_quantities
    }

# Test it
quality = check_data_quality(df)
print('Data quality report:')
for key, value in quality.items():
    status = 'OK' if value == 0 else f'WARNING: {value} found'
    print(f'  {key}: {status}')



# Clear outputs -> Commit: 'Section 1: Add check_data_quality() - [Name]'

In [ ]:
# Section 1, Cell 3: Unique value counts

def dataset_summary(dataframe):
    """Summarize unique values and ranges in the dataset.

    Args:
        dataframe: the sales DataFrame

    Returns:
        dict with keys:
          'num_products': int, count of unique products
          'num_regions': int, count of unique regions
          'num_reps': int, count of unique sales reps
          'revenue_range': tuple (min_revenue, max_revenue) per order

    >>> import pandas as pd
    >>> test_df = pd.DataFrame({'Product': ['A','B','A'], 'Region': ['N','S','N'],
    ...                         'SalesRep': ['X','X','Y'], 'Revenue': [100, 50, 200]})
    >>> s = dataset_summary(test_df)
    >>> s['num_products']
    2
    >>> s['num_reps']
    2
    """
    num_products = dataframe['Product'].nunique()
    num_regions = dataframe['Region'].nunique()
    num_reps = dataframe['SalesRep'].nunique()
    revenue_range = (dataframe['Revenue'].min(), dataframe['Revenue'].max())

    return {
        'num_products': num_products,
        'num_regions': num_regions,
        'num_reps': num_reps,
        'revenue_range': revenue_range
    }


summary = dataset_summary(df)
print('Dataset summary:')
for key, value in summary.items():
    print(f'  {key}: {value}')

# Clear outputs -> Commit: 'Section 1: Add dataset_summary() - [Name]'

Section 2 - Linh Vu

The Laptop generated the highest revenue at $8,400.00, significantly outperforming all other products. The next highest revenue came from Monitors ($2,994.00), followed by Webcams ($2,079.74), Keyboards ($1,799.64), and Headsets ($1,529.83).

This indicates that laptops are the primary revenue driver and may deserve continued marketing focus, bundling opportunities, or premium positioning to maximize profitability.

In [ ]:
# Section 2, Cell 1: Revenue by product

def revenue_by_product(dataframe):
    """Calculate total revenue for each product.

    Args:
        dataframe: sales DataFrame with 'Product' and 'Revenue' columns

    Returns:
        dict mapping product name to total revenue (float), sorted descending
        e.g. {'Laptop': 8400.00, 'Monitor': 1996.00, ...}

    >>> import pandas as pd
    >>> test = pd.DataFrame({'Product': ['A','B','A'], 'Revenue': [100.0, 50.0, 200.0]})
    >>> revenue_by_product(test)
    {'A': 300.0, 'B': 50.0}
    """
    return dataframe.groupby('Product')['Revenue'].sum().to_dict()


by_product = revenue_by_product(df)
print('Revenue by product:')
for product, rev in by_product.items():
    print(f'  {product}: ${rev:,.2f}')

# Clear outputs -> Commit: 'Section 2: Add revenue_by_product() - [Name]'

In [ ]:
# Section 2, Cell 2: Revenue by region

def revenue_by_region(dataframe):
    """Calculate total revenue for each region.

    Args:
        dataframe: sales DataFrame with 'Region' and 'Revenue' columns

    Returns:
        dict mapping region name to total revenue (float), sorted descending

    >>> import pandas as pd
    >>> test = pd.DataFrame({'Region': ['North','South','North'], 'Revenue': [500.0, 200.0, 300.0]})
    >>> revenue_by_region(test)
    {'North': 800.0, 'South': 200.0}
    """
    totals = dataframe.groupby('Region')['Revenue'].sum()
    # return as a dict sorted by revenue descending
    return dict(totals.sort_values(ascending=False))


by_region = revenue_by_region(df)
print('Revenue by region:')
for region, rev in by_region.items():
    print(f'  {region}: ${rev:,.2f}')

# Clear outputs -> Commit: 'Section 2: Add revenue_by_region() - [Name]'

In [ ]:
# Section 2, Cell 3: Top product by revenue

def top_revenue_item(revenue_dict):
    """Return the name and revenue of the highest-earning item.

    Args:
        revenue_dict: dict mapping name to revenue (output of revenue_by_product
                      or revenue_by_region)

    Returns:
        tuple: (name, revenue)

    >>> top_revenue_item({'Laptop': 8400.0, 'Monitor': 1996.0, 'Headset': 719.92})
    ('Laptop', 8400.0)
    """
    if not revenue_dict:
        return None, 0.0
    top_item = max(revenue_dict.items(), key=lambda x: x[1])
    return top_item


top_product = top_revenue_item(revenue_by_product(df))
top_region  = top_revenue_item(revenue_by_region(df))
print(f'Top product: {top_product[0]} (${top_product[1]:,.2f})')
print(f'Top region:  {top_region[0]} (${top_region[1]:,.2f})')

# Clear outputs -> Commit: 'Section 2: Add top_revenue_item() - [Name]'

In [ ]:
#section 3 cell 1 correction
# Section 3, Cell 1: Revenue by month

def revenue_by_month(dataframe):
    """Calculate total revenue for each calendar month."""

    # Extract month numbers and month names
    dataframe['Month'] = dataframe['Date'].dt.month
    dataframe['MonthName'] = dataframe['Date'].dt.month_name()

    # Group by month number (to keep chronological order)
    monthly_rev = dataframe.groupby(['Month', 'MonthName'])['Revenue'].sum()

    # Convert to ordered dict: { 'January': 3299.95, 'February': ... }
    ordered = {name: monthly_rev.loc[(num, name)] for num, name in monthly_rev.index}

    return ordered


# Run it
monthly = revenue_by_month(df)
print('Monthly revenue:')
for month, rev in monthly.items():
    print(f'  {month}: ${rev:,.2f}')

    # Clear outputs -> Commit: 'Section 3: Add revenue_by_month() - [Name]'

In [ ]:
# Section 3, Cell 2: Order count by month

def orders_by_month(dataframe):
    """Count the number of orders placed in each calendar month."""

    # Extract month numbers and month names
    dataframe['Month'] = dataframe['Date'].dt.month
    dataframe['MonthName'] = dataframe['Date'].dt.month_name()

    # Group by month number (keeps chronological order)
    monthly_counts = dataframe.groupby(['Month', 'MonthName'])['OrderID'].count()

    # Build ordered dictionary: {'January': 2, 'February': 1, ...}
    ordered = {name: monthly_counts.loc[(num, name)] for num, name in monthly_counts.index}

    return ordered


# Run it
monthly_orders = orders_by_month(df)
print('Orders per month:')
for month, count in monthly_orders.items():
    print(f'  {month}: {count} orders')
    # Clear outputs -> Commit: 'Section 3: Add orders_by_month() - [Name]'

In [ ]:
# Section 3, Cell 3: Best and worst month

def month_comparison(monthly_revenue):
    """Identify the best and worst revenue months."""

    # Convert dict to list of (month, revenue) pairs
    months = list(monthly_revenue.keys())
    revenues = list(monthly_revenue.values())

    # Best and worst months
    best_month = months[revenues.index(max(revenues))]
    best_revenue = max(revenues)

    worst_month = months[revenues.index(min(revenues))]
    worst_revenue = min(revenues)

    # Growth percentage from first to last month
    first = revenues[0]
    last = revenues[-1]
    growth_pct = ((last - first) / first) * 100 if first != 0 else 0

    return {
        'best_month': best_month,
        'best_revenue': best_revenue,
        'worst_month': worst_month,
        'worst_revenue': worst_revenue,
        'growth_pct': growth_pct
    }


# Run it
comparison = month_comparison(monthly)
print(f"Best month:  {comparison['best_month']} (${comparison['best_revenue']:,.2f})")
print(f"Worst month: {comparison['worst_month']} (${comparison['worst_revenue']:,.2f})")
print(f"Q1 growth:   {comparison['growth_pct']:.1f}% from Jan to Mar")

# Clear outputs -> Commit: 'Section 3: Add month_comparison() - [Name]